In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"

%load_ext autoreload
%autoreload 2

In [2]:
import ipdb
from Data import *
from address import *
from modelGen import *
from utils import cf_eval, cleanup_gpu
from models import CausalCondLatentCF, latentCFpp, PrototypeLatentCF
from models import CondLatentCF # Note: get the vanilla CondLatentCF
import numpy as np
import random 
from tensorflow import random as tf_random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
import argparse
import os
import pandas as pd


In [3]:
SEED = 23

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf_random.set_seed(SEED)


CF_METHODS = {  
    "LatentCFpp": latentCFpp,
    "Prototype": PrototypeLatentCF,
    "CausalCACTUS": CausalCondLatentCF,
    "CACTUS": CondLatentCF
}

In [4]:
# Experimentos que a realizar
EXP = [
    {
        "name": "LatentCF++",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_AE/config.json",
        "CFmethod": "LatentCFpp",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 100,
        "tol": 0.001,
        "target_prob": 0.5,
        "learning_rate": 0.1
    },
    {
        "name": "Prototype C",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 0.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "Prototype D",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 100.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "CACTUS",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        # "dynamicAlpha": True
        "dynamicAlpha": False
    },
    {
        "name": "CausalCACTUS",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_CACTUS/config.json",
        "CFmethod": "CausalCACTUS",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": True
        # "dynamicAlpha": False
    }
]

In [5]:
import re
from collections import defaultdict
# TODO: make a function to convert the causal paths for all the test samples
def build_causal_index_map(input_features, graph_str):
    """
    Returns:
        child_to_parents: dict {child_idx: [parent_idx, ...]}
        parent_to_children: dict {parent_idx: [child_idx, ...]}
        feat2idx: dict {feature_name: index}
    """

    # Feature name to index
    feat2idx = {f: i for i, f in enumerate(input_features)}

    # Extract edges using regex
    edges = re.findall(r'(\w+)\s*->\s*(\w+)', graph_str)

    child_to_parents = defaultdict(list)
    parent_to_children = defaultdict(list)

    for parent, child in edges:

        if parent not in feat2idx or child not in feat2idx:
            raise ValueError(f"Feature in graph not found in dataset: {parent} or {child}")

        p_idx = feat2idx[parent]
        c_idx = feat2idx[child]

        child_to_parents[c_idx].append(p_idx)
        parent_to_children[p_idx].append(c_idx)

    return dict(child_to_parents), dict(parent_to_children), feat2idx

# credit_rex5_constraints_min.dot from downsampled
graph_str1 = """
strict digraph {
NumberRealEstateLoansOrLines;
NumberOfTimes90DaysLate;
NumberOfDependents;
age;
DebtRatio;
NumberOfTime3059DaysPastDueNotWorse;
NumberOfOpenCreditLinesAndLoans;
RevolvingUtilizationOfUnsecuredLines;
MonthlyIncome;
NumberOfTime6089DaysPastDueNotWorse;
NumberOfTimes90DaysLate -> NumberOfOpenCreditLinesAndLoans;
NumberOfTimes90DaysLate -> NumberRealEstateLoansOrLines;
NumberOfTimes90DaysLate -> RevolvingUtilizationOfUnsecuredLines;
NumberOfTimes90DaysLate -> MonthlyIncome;
NumberOfTime3059DaysPastDueNotWorse -> NumberOfOpenCreditLinesAndLoans;
NumberOfTime3059DaysPastDueNotWorse -> NumberRealEstateLoansOrLines;
NumberOfTime3059DaysPastDueNotWorse -> MonthlyIncome;
NumberOfOpenCreditLinesAndLoans -> DebtRatio;
RevolvingUtilizationOfUnsecuredLines -> NumberOfOpenCreditLinesAndLoans;
MonthlyIncome -> NumberOfOpenCreditLinesAndLoans;
}
"""

# credit_rex6_constraints_mod.dot from downsampled
graph_str2 = """ 
strict digraph {
NumberRealEstateLoansOrLines;
NumberOfTimes90DaysLate;
NumberOfDependents;
age;
DebtRatio;
NumberOfTime3059DaysPastDueNotWorse;
NumberOfOpenCreditLinesAndLoans;
RevolvingUtilizationOfUnsecuredLines;
MonthlyIncome;
NumberOfTime6089DaysPastDueNotWorse;
NumberRealEstateLoansOrLines -> DebtRatio;
NumberOfOpenCreditLinesAndLoans -> DebtRatio;
RevolvingUtilizationOfUnsecuredLines -> NumberOfOpenCreditLinesAndLoans;
MonthlyIncome -> NumberOfOpenCreditLinesAndLoans;
}
"""

In [6]:
# # CF generation
# child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
#     data.features_lbls,
#     # graph_str
#     graph_str1
# )
# print("Feature → index:")
# print(feat2idx)
# print("\nChild → Parents (index form):")
# print(child_to_parents_dict)

## Graph 1 (Minimum priors)

In [7]:
N = 50 
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [12, 23, 34, 45, 56]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str1
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              
      
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)





********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Class Good Standing: 8334.0 samples
Class Default Risk: 8334.0 samples
Age above 50 att: 7061
Dependents more than one: 8256
----------------------------------------------------------------------------------------------------




Building model


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 8)]               0         
                                                                 
 dense (Dense)               (None, 64)                576       
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                 

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


restoring weights of model AE: GIVECREDIT_AE
Feature → index:
{'RevolvingUtilizationOfUnsecuredLines': 0, 'NumberOfTime3059DaysPastDueNotWorse': 1, 'DebtRatio': 2, 'MonthlyIncome': 3, 'NumberOfOpenCreditLinesAndLoans': 4, 'NumberOfTimes90DaysLate': 5, 'NumberRealEstateLoansOrLines': 6, 'NumberOfTime6089DaysPastDueNotWorse': 7}

Child → Parents (index form):
{4: [5, 1, 0, 3], 6: [5, 1], 0: [5], 3: [5, 1], 2: [4]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 model_2 (Functional)        (None, 8)                 4648      
                                                                 
 model (Functional)          (None, 2)                 35330     
                                                                 
Total par

...

[1 1 0 1 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 1 0 0 1 0 1 0 1 0 1 1 0
 1 1 0 1 0 0 0 0 1 1 1 0 0]
[1 1 0 1 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 1 1 0 1 1 0 1 0 1 0 1 1 1 1 0
 1 1 0 1 0 0 0 0 1 1 1 1 0]
0.92
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 3243
Unique rows: 3216
After np.unique(): subset size: 3216
cf_samples_ shape: (12, 8)
X_ shape: (3216, 8)
lof_cf min/max: 0.9675698090240107 1.2338810235143767
lof_train min/max: 0.9465570652688365 3.282124843560917
mean train LOF: 1.1444881201316588
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 4393
Unique rows: 4379
After np.unique(): subset size: 4379
cf_samples_ shape: (15, 8)
X_ shape: (4379, 8)
lof_cf min/max: 0.9457660928791086 1.230223561232869
lof_train min/max: 0.9533726709159543 61.731034094851054
mean train LOF: 1.1579012060688825
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 3563
Unique rows: 3553
After np.unique(): subset size: 3553
cf_samples_ shape: (14, 8)
X_ shape:

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(


[1 1 0 0 0 0 0 0 0 0 1 0 1 1 0 1 1 0 1 1 0 0 1 1 0 1 0 0 1 1 0 0 0 1 0 0 0
 1 1 0 1 1 1 1 0 1 1 1 1 1]
[1 1 0 0 0 0 0 0 1 0 1 0 1 1 0 1 1 0 1 1 0 1 1 1 0 1 0 0 1 1 1 0 0 1 0 1 0
 1 1 0 1 1 1 1 0 1 1 1 1 1]
0.92
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 3243
Unique rows: 3216
After np.unique(): subset size: 3216
cf_samples_ shape: (14, 8)
X_ shape: (3216, 8)
lof_cf min/max: 0.966730782073915 1.4975427152938676
lof_train min/max: 0.9465570652688365 3.282124843560917
mean train LOF: 1.1444881201316588
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 4393
Unique rows: 4379
After np.unique(): subset size: 4379
cf_samples_ shape: (17, 8)
X_ shape: (4379, 8)
lof_cf min/max: 0.9518937309530829 1.175531961804953
lof_train min/max: 0.9533726709159543 61.731034094851054
mean train LOF: 1.1579012060688825
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 3563
Unique rows: 3553
After np.unique(): subset size: 3553
cf_samples_ shape: (12, 8)
X_ shape: 

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(


[1 0 1 1 1 1 1 1 1 0 0 0 0 1 0 1 0 1 1 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 1 0 1
 1 0 1 0 1 0 0 0 0 0 1 1 1]
[1 0 1 0 0 1 1 1 0 1 0 0 0 1 1 1 0 1 1 0 0 0 1 0 0 1 0 1 0 1 0 0 0 1 1 0 1
 1 0 1 0 1 0 0 1 0 1 1 1 1]
0.82
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 3243
Unique rows: 3216
After np.unique(): subset size: 3216
cf_samples_ shape: (9, 8)
X_ shape: (3216, 8)
lof_cf min/max: 0.9872142963152252 1.7190283387654206
lof_train min/max: 0.9465570652688365 3.282124843560917
mean train LOF: 1.1444881201316588
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 4393
Unique rows: 4379
After np.unique(): subset size: 4379
cf_samples_ shape: (18, 8)
X_ shape: (4379, 8)
lof_cf min/max: 0.968216923923668 1.3721144140196038
lof_train min/max: 0.9533726709159543 61.731034094851054
mean train LOF: 1.1579012060688825
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 3563
Unique rows: 3553
After np.unique(): subset size: 3553
cf_samples_ shape: (15, 8)
X_ shape: 

In [8]:
child_to_parents_dict

{4: [5, 1, 0, 3], 6: [5, 1], 0: [5], 3: [5, 1], 2: [4]}

In [9]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                      Validity          LOF  Compactness    Proximity  \
Dataset    Model                                                              
GIVECREDIT CACTUS        0.69 ± 0.07  1.16 ± 0.02  0.09 ± 0.03  0.52 ± 0.03   
           CausalCACTUS  0.89 ± 0.04  1.11 ± 0.01  0.25 ± 0.04  0.41 ± 0.02   
           LatentCF++      1.0 ± 0.0   1.41 ± 0.1  0.05 ± 0.01   0.71 ± 0.1   
           Prototype C     1.0 ± 0.0  2.02 ± 0.13  0.04 ± 0.02  1.12 ± 0.11   
           Prototype D     1.0 ± 0.0  1.75 ± 0.11  0.04 ± 0.01  1.07 ± 0.04   

Metric                  Causal Validity (Hard) Causal Validity (Soft)  \
Dataset    Model                                                        
GIVECREDIT CACTUS                  0.35 ± 0.02            0.78 ± 0.01   
           CausalCACTUS            0.34 ± 0.04            0.78 ± 0.04   
           LatentCF++              0.25 ± 0.03             0.7 ± 0.02   
           Prototype C             0.53 ± 0.04            0.85 ± 0.03   
           Prototype D             0.45 ± 0.11            0.82 ± 0.03   

Metric                  Causal Compact Validity inlier_fraction  
Dataset    Model                                                 
GIVECREDIT CACTUS                   0.39 ± 0.02     0.95 ± 0.03  
           CausalCACTUS             0.45 ± 0.04     0.97 ± 0.02  
           LatentCF++               0.33 ± 0.03     0.75 ± 0.08  
           Prototype C               0.3 ± 0.02     0.33 ± 0.08  
           Prototype D               0.3 ± 0.01     0.43 ± 0.07

## Graph 2 (Moderate priors)

In [ ]:
N = 50 
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [11, 22, 33, 44, 55]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str2
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              
      
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)





********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Class Good Standing: 8334.0 samples
Class Default Risk: 8334.0 samples
Age above 50 att: 7107
Dependents more than one: 8235
----------------------------------------------------------------------------------------------------




Building model
Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Building model
Building model: AE [gen]
Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 8)]               0         
                                                                 
 model_1 (Functional)        (None, 16)                4656      
                                                                 
 model_2 (Functional)        (None, 8)                 4648      
                                                                 
Total params: 9,304
Trainable params: 9,304
Non-trainable params: 0
_________________________________________________________________
restoring weights of model AE: GIVECREDIT_AE


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Feature → index:
{'RevolvingUtilizationOfUnsecuredLines': 0, 'NumberOfTime3059DaysPastDueNotWorse': 1, 'DebtRatio': 2, 'MonthlyIncome': 3, 'NumberOfOpenCreditLinesAndLoans': 4, 'NumberOfTimes90DaysLate': 5, 'NumberRealEstateLoansOrLines': 6, 'NumberOfTime6089DaysPastDueNotWorse': 7}

Child → Parents (index form):
{2: [6, 4], 4: [0, 3]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 model_2 (Functional)        (None, 8)                 4648      
                                                                 
 model (Functional)          (None, 2)                 35330     
                                                                 
Total params: 39,978
Trainable params: 39,210
Non-trainable params: 768
_______________

...

[1 1 1 0 0 0 1 1 1 0 1 1 1 0 1 0 0 0 0 1 1 1 1 0 1 1 1 1 0 0 1 1 0 1 0 0 1
 0 1 1 0 1 1 0 0 1 0 0 0 0]
[1 1 1 0 0 0 1 1 1 0 1 1 1 0 1 0 0 1 0 1 1 1 1 0 1 1 1 1 0 1 1 1 0 1 0 0 1
 1 1 1 0 1 1 0 0 1 0 0 0 0]
0.94
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 3217
Unique rows: 3195
After np.unique(): subset size: 3195
cf_samples_ shape: (11, 8)
X_ shape: (3195, 8)
lof_cf min/max: 0.9828567574405701 1.7350546834457192
lof_train min/max: 0.9500319961567154 4.697967518621444
mean train LOF: 1.1470414414914387
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 4431
Unique rows: 4413
After np.unique(): subset size: 4413
cf_samples_ shape: (11, 8)
X_ shape: (4413, 8)
lof_cf min/max: 0.9659703131554692 1.421432008043142
lof_train min/max: 0.9444151923412731 7.6460548321558175
mean train LOF: 1.1380723585473191
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 3592
Unique rows: 3584
After np.unique(): subset size: 3584
cf_samples_ shape: (14, 8)
X_ shape:

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(


[1 0 1 0 1 0 0 1 0 1 1 1 1 0 0 1 0 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 0
 1 1 0 0 0 1 0 0 0 1 0 0 1]
[1 0 1 0 1 1 0 0 0 1 1 1 1 0 0 1 0 0 0 0 1 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0
 1 1 0 0 1 1 0 1 0 1 1 0 1]
0.84
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 3217
Unique rows: 3195
After np.unique(): subset size: 3195
cf_samples_ shape: (21, 8)
X_ shape: (3195, 8)
lof_cf min/max: 0.9691630957873529 1.5529803819005537
lof_train min/max: 0.9500319961567154 4.697967518621444
mean train LOF: 1.1470414414914387
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 4431
Unique rows: 4413
After np.unique(): subset size: 4413
cf_samples_ shape: (9, 8)
X_ shape: (4413, 8)
lof_cf min/max: 0.9422277080202868 1.62231927018684
lof_train min/max: 0.9444151923412731 7.6460548321558175
mean train LOF: 1.1380723585473191
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 3592
Unique rows: 3584
After np.unique(): subset size: 3584
cf_samples_ shape: (13, 8)
X_ shape: (

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(


[0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 1 1 0 0 1 1 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0
 1 1 0 0 1 0 1 0 0 1 0 1 0]
[0 1 1 0 1 0 0 0 1 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 1 0 0 1 1 0 0 1 1 0 0
 1 1 0 0 1 0 1 0 0 1 0 1 0]
0.88
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 3217
Unique rows: 3195
After np.unique(): subset size: 3195
cf_samples_ shape: (14, 8)
X_ shape: (3195, 8)
lof_cf min/max: 0.9654472529413438 1.1591560264950738
lof_train min/max: 0.9500319961567154 4.697967518621444
mean train LOF: 1.1470414414914387
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 4431
Unique rows: 4413
After np.unique(): subset size: 4413
cf_samples_ shape: (19, 8)
X_ shape: (4413, 8)
lof_cf min/max: 0.966472372379683 1.304448421058649
lof_train min/max: 0.9444151923412731 7.6460548321558175
mean train LOF: 1.1380723585473191
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 3592
Unique rows: 3584
After np.unique(): subset size: 3584
cf_samples_ shape: (11, 8)
X_ shape: 

In [13]:
child_to_parents_dict

{2: [6, 4], 4: [0, 3]}

In [14]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                      Validity          LOF  Compactness    Proximity  \
Dataset    Model                                                              
GIVECREDIT CACTUS         0.6 ± 0.07  1.13 ± 0.03  0.09 ± 0.02  0.52 ± 0.03   
           CausalCACTUS  0.87 ± 0.04  1.12 ± 0.03  0.26 ± 0.04  0.42 ± 0.02   
           LatentCF++      1.0 ± 0.0   1.4 ± 0.03   0.06 ± 0.0  0.66 ± 0.07   
           Prototype C     1.0 ± 0.0  1.99 ± 0.15  0.04 ± 0.02  1.23 ± 0.06   
           Prototype D     1.0 ± 0.0   1.71 ± 0.1  0.04 ± 0.01   1.1 ± 0.02   

Metric                  Causal Validity (Hard) Causal Validity (Soft)  \
Dataset    Model                                                        
GIVECREDIT CACTUS                  0.86 ± 0.04            0.93 ± 0.02   
           CausalCACTUS            0.86 ± 0.05            0.93 ± 0.03   
           LatentCF++               0.8 ± 0.05             0.9 ± 0.02   
           Prototype C              0.9 ± 0.05            0.95 ± 0.03   
           Prototype D             0.94 ± 0.03            0.97 ± 0.02   

Metric                  Causal Compact Validity inlier_fraction  
Dataset    Model                                                 
GIVECREDIT CACTUS                   0.47 ± 0.03     0.96 ± 0.03  
           CausalCACTUS             0.53 ± 0.02     0.96 ± 0.02  
           LatentCF++               0.42 ± 0.01     0.73 ± 0.06  
           Prototype C              0.34 ± 0.02     0.27 ± 0.07  
           Prototype D              0.36 ± 0.02     0.46 ± 0.08